## Generate benchmarking figures

#### Generate legends

In [1]:
import matplotlib.pyplot as plt
import matplotlib as mpl
from matplotlib import font_manager as fm
from matplotlib.patches import Patch
from pathlib import Path

font_path = "../resources/fonts/InterVariable.ttf"
fm.fontManager.addfont(font_path)
prop = fm.FontProperties(fname=font_path)
mpl.rcParams['font.family'] = prop.get_name()

def save_legend(boxed: bool = True, show_title: bool = True):

    color_map = {
        "FuncVEP": "#b21e35",
        "Clinical-trained": "goldenrod",
        "Population-free": "#98e5a5",
        "Population-tuned": "#23a4a6",
    }

    def _make_handles(linewidth=0.4):
        return [
            Patch(facecolor=color_map[cat], edgecolor='black', linewidth=linewidth, label=cat)
            for cat in color_map
        ]

    def _save(ax, out_path):
        out_path.parent.mkdir(parents=True, exist_ok=True)
        plt.tight_layout()
        fig = ax.figure
        fig.savefig(out_path, dpi=300, bbox_inches="tight")
        plt.close(fig)
        print(f"Saved legend to: {out_path}")

    fig, ax = plt.subplots(figsize=(2.2, 1))
    ax.axis("off")
    handles = _make_handles(linewidth=0.4)
    legend = ax.legend(
        handles=handles,
        loc="center",
        fontsize=7,
        title="Category" if show_title else None,
        title_fontsize=8,
        ncol=1,
        frameon=boxed,
        edgecolor="black" if boxed else None,
        labelspacing=0.5,
        borderpad=0.6,
        handlelength=1.2,
        handletextpad=0.6,
        borderaxespad=0.0
    )
    if show_title:
        legend.get_title().set_position((0, 4))
    _save(ax, Path("../results/figures/benchmarks/legend_vertical_box.png"))


In [2]:
save_legend(boxed=False, show_title=False)

Saved legend to: ..\results\figures\benchmarks\legend_vertical_box.png


#### Generate bar plots

In [3]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import re
from matplotlib.patches import Patch
from matplotlib import font_manager as fm
import matplotlib as mpl
from pathlib import Path

font_path = "../resources/fonts/InterVariable.ttf"
fm.fontManager.addfont(font_path)
prop = fm.FontProperties(fname=font_path)
mpl.rcParams["font.family"] = prop.get_name()

def generate_benchmark_figure(
    dataset_name="functional",
    include_legend=True,
    remove_clinvep=True,
    len_x=None,
    filter_glm=False,
    linewidth=0.5,
    gap_frac=0.25,
    outer_pad_frac=1,
    bar_height=0.3,
):
    """
    dataset_name ∈ {
        "cancer", "clinical", "dd", "functional", # AUC-based benchmark
        "mave", "proteingym" # RankScore-based benchmarks
    }
    """

    if dataset_name in ["cancer", "clinical", "dd", "functional"]:
        data_file = f"../results/benchmarks/{dataset_name}_benchmark.txt"
        score_col = "AUC"
        std_col = "AUC_boot_std"
        x_label = "AUC"
        default_len_x = 1.0

        if dataset_name == "cancer":
            plot_title = "Performance on cancer hotspots"
        elif dataset_name == "clinical":
            plot_title = "Performance on clinical data"
        elif dataset_name == "dd":
            plot_title = "Performance on developmental disorder variants"
        elif dataset_name == "functional":
            plot_title = "Performance on functional data"
        else:
            plot_title = f"Performance on {dataset_name.upper()} variants"

    elif dataset_name in ["mave", "proteingym"]:
        data_file = f"../results/benchmarks/{dataset_name}_benchmark.txt"
        if dataset_name == "mave":
            plot_title = "Performance on MAVE datasets"
        else:
            plot_title = "Performance on ProteinGym DMS datasets"

        score_col = "RankScore"
        std_col = "RankScoreSD"
        x_label = "Average win rate (%)"
        default_len_x = 1.0

    else:
        raise ValueError(f"Unknown dataset_name: {dataset_name}")

    len_x = len_x if len_x is not None else default_len_x
    features_file = "../resources/feature_lists/all_columns.txt"
    output_figure_path = Path(
        f"../results/figures/benchmarks/{dataset_name}_performance_plot.png"
    )

    df = pd.read_csv(data_file, sep="\t")

    if "VEP" not in df.columns:
        raise ValueError(f"'VEP' column not found in {data_file}")

    df = df.dropna(subset=[score_col]).copy()

    features_df = pd.read_csv(features_file, sep="\t")
    if "Name" not in features_df.columns:
        raise ValueError("Expected 'Name' column in feature metadata file.")

    features_df = features_df.rename(columns={"Name": "VEP"})

    if filter_glm:
        glm_mask = df["VEP"].str.startswith("glm_")
        if glm_mask.any():
            best_glm = df.loc[glm_mask].nlargest(1, score_col)
            glm_to_keep = best_glm["VEP"].unique()
            df = df[(~glm_mask) | (df["VEP"].isin(glm_to_keep))].copy()

    if "Category" not in features_df.columns:
        raise ValueError("Expected 'Category' column in feature metadata file.")

    df = df.merge(features_df[["VEP", "Category"]], on="VEP", how="left")

    if remove_clinvep:
        df = df[~df["VEP"].str.startswith("ClinVEP_")].copy()

    def clean_vep_name(vep):
        vep = re.sub(r"^glm_", "", vep)
        vep = re.sub(r"_score$", "", vep)
        return vep.replace("___", "-").replace("__", "-").replace("_", "-")

    df["VEP_clean"] = df["VEP"].apply(clean_vep_name)

    category_rename = {
        "FuncVEP": "FuncVEP",
        "Clinical-Trained Meta Predictor": "Clinical-trained",
        "Clinical-Trained Single Predictor": "Clinical-trained",
        "Population-Free": "Population-free",
        "Population-Tuned": "Population-tuned",
    }
    df["PlotCategory"] = df["Category"].map(category_rename).fillna(df["Category"])

    color_map = {
        "FuncVEP": "#b21e35",
        "Clinical-trained": "goldenrod",
        "Population-free": "#98e5a5",
        "Population-tuned": "#23a4a6",
    }
    df["Color"] = df["PlotCategory"].map(color_map).fillna("white")

    df = df.sort_values(score_col, ascending=True).reset_index(drop=True)
    n_bars = len(df)

    gap_ref = 0.5
    gap = max(0.05, gap_ref * gap_frac)
    step = bar_height + gap
    y_pos = np.arange(n_bars) * step
    outer_pad = bar_height * outer_pad_frac

    plt.rcParams.update({
        "font.size": 4.5,
    })

    fig_height = max(2.0, 0.28 * n_bars * (step / 1.0))
    fig, ax = plt.subplots(figsize=(6, fig_height))

    if score_col == "AUC":
        widths = df[score_col] - 0.5
        lefts = np.full_like(widths, 0.5)
        x_min = 0.5
    else:
        widths = df[score_col]
        lefts = np.zeros_like(widths)
        x_min = 0.0

    ax.barh(
        y=y_pos,
        width=widths,
        left=lefts,
        color=df["Color"],
        height=bar_height,
        edgecolor="black",
        linewidth=linewidth,
    )

    if std_col in df.columns:
        x_vals = df[score_col].values
        x_err = df[std_col].values

        ax.errorbar(
            x=x_vals,
            y=y_pos,
            xerr=x_err,
            fmt="none",
            ecolor="black",
            elinewidth=linewidth,
            capsize=0,
            alpha=1,
        )

    ax.set_yticks(y_pos)
    ax.set_yticklabels(df["VEP_clean"])
    ax.set_ylim(y_pos.min() - outer_pad, y_pos.max() + outer_pad)

    ax.set_xlim([x_min, len_x])

    if score_col == "RankScore":
        formatter = mpl.ticker.FuncFormatter(lambda v, pos: f"{v*100:.0f}")
        ax.xaxis.set_major_formatter(formatter)

    ax.set_xlabel(x_label, fontsize=6)
    ax.set_title(plot_title, fontsize=8, loc="center", pad=10)

    if include_legend:
        present_cats = [c for c in color_map.keys() if c in df["PlotCategory"].values]
        handles = [
            Patch(facecolor=color_map[c], label=c, edgecolor="black", linewidth=linewidth)
            for c in present_cats
        ]
        if handles:
            ax.legend(
                handles=handles,
                loc="lower right",
                fontsize=7,
                frameon=True,
                edgecolor="gray",
            )

    plt.tight_layout()
    output_figure_path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(output_figure_path, dpi=300)
    plt.close(fig)
    print(f"Figure saved to: {output_figure_path}")


In [4]:
generate_benchmark_figure("clinical", include_legend=True, filter_glm=False, len_x = 1.0)
generate_benchmark_figure("functional", include_legend=True, filter_glm=False, len_x=0.95)
generate_benchmark_figure("dd", include_legend=True, len_x=0.90)
generate_benchmark_figure("cancer", include_legend=True, len_x=0.95)
generate_benchmark_figure("mave", include_legend=True, filter_glm=False, len_x=1)
generate_benchmark_figure("proteingym", include_legend=True, filter_glm=False, len_x=1)

Figure saved to: ..\results\figures\benchmarks\clinical_performance_plot.png
Figure saved to: ..\results\figures\benchmarks\functional_performance_plot.png
Figure saved to: ..\results\figures\benchmarks\dd_performance_plot.png
Figure saved to: ..\results\figures\benchmarks\cancer_performance_plot.png
Figure saved to: ..\results\figures\benchmarks\mave_performance_plot.png
Figure saved to: ..\results\figures\benchmarks\proteingym_performance_plot.png


#### Generate filtered bar plots

In [5]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import re
import matplotlib as mpl
from matplotlib.patches import Patch
from matplotlib import font_manager as fm
from pathlib import Path

font_path = "../resources/fonts/InterVariable.ttf"
fm.fontManager.addfont(font_path)
prop = fm.FontProperties(fname=font_path)
mpl.rcParams['font.family'] = prop.get_name()

def generate_filtered_benchmark_figure(
    dataset_name="functional",
    bold_keep_veps=False,
    remove_clinvep=True,
    len_x=None,
    linewidth=1,
    gap_frac=0.25,
    fig_height_per_bar=0.6,
    outer_pad_frac=0.5,
    veps_per_category=3,
):
    if dataset_name in ["cancer", "clinical", "dd", "functional"]:
        data_file = f"../results/benchmarks/{dataset_name}_benchmark.txt"
        score_col = "AUC"
        std_col = "AUC_boot_std"
        x_label = "AUC"
        default_len_x = 1.0

        if dataset_name == "cancer":
            plot_title = "Performance on cancer hotspots"
        elif dataset_name == "clinical":
            plot_title = "Performance on clinical data"
        elif dataset_name == "dd":
            plot_title = "Performance on developmental disorder variants"
        elif dataset_name == "functional":
            plot_title = "Performance on functional data"
        else:
            plot_title = f"Performance on {dataset_name.upper()} variants"

    elif dataset_name in ["mave", "proteingym"]:
        data_file = f"../results/benchmarks/{dataset_name}_benchmark.txt"
        if dataset_name == "mave":
            plot_title = "Performance on MAVE datasets"
        else:
            plot_title = "Performance on ProteinGym DMS datasets"

        score_col = "RankScore"
        std_col = "RankScoreSD"
        x_label = "Average win rate (%)"
        default_len_x = 1.0

    else:
        raise ValueError(f"Unknown dataset_name: {dataset_name}")

    len_x = len_x if len_x is not None else default_len_x
    features_file = "../resources/feature_lists/all_columns.txt"
    output_figure_path = Path(
        f"../results/figures/benchmarks/{dataset_name}_filtered_performance_plot.png"
    )

    df = pd.read_csv(data_file, sep="\t")

    if "VEP" not in df.columns:
        raise ValueError(f"'VEP' column not found in {data_file}")

    df = df.dropna(subset=[score_col]).copy()

    features_df = pd.read_csv(features_file, sep="\t")
    if "Name" not in features_df.columns:
        raise ValueError("Expected 'Name' column in feature metadata file.")
    if "Category" not in features_df.columns:
        raise ValueError("Expected 'Category' column in feature metadata file.")

    features_df = features_df.rename(columns={"Name": "VEP"})

    df = df.merge(features_df[["VEP", "Category"]], on="VEP", how="left")

    # Optionally remove ClinVEP models
    if remove_clinvep:
        df = df[~df["VEP"].str.startswith("ClinVEP_")].copy()

    if df.empty:
        raise ValueError(
            f"No VEPs remaining for dataset '{dataset_name}' after filtering. "
            f"Check remove_clinvep settings and metadata."
        )

    category_rename = {
        "FuncVEP": "FuncVEP",
        "Clinical-Trained Meta Predictor": "Clinical-trained",
        "Clinical-Trained Single Predictor": "Clinical-trained",
        "Population-Free": "Population-free",
        "Population-Tuned": "Population-tuned",
    }
    df["PlotCategory"] = df["Category"].map(category_rename).fillna(df["Category"])

    color_map = {
        "FuncVEP": "#b21e35",
        "Clinical-trained": "goldenrod",
        "Population-free": "#98e5a5",
        "Population-tuned": "#23a4a6",
    }

    always_keep = {
        "FuncVEP_CTI", "FuncVEP_CTE", "FuncVEP_SP",
        "ClinVEP_CTI", "ClinVEP_CTE", "ClinVEP_SP",
    }

    remaining_df = df[~df["VEP"].isin(always_keep)].copy()

    # Within each category, keep top-N VEPs by score
    best_per_category = (
        remaining_df.sort_values(score_col, ascending=False)
        .groupby("PlotCategory", dropna=False)
        .head(veps_per_category)
    )

    selected_df = pd.concat(
        [
            df[df["VEP"].isin(always_keep)],
            best_per_category,
        ]
    ).drop_duplicates(subset="VEP")

    def clean_vep_name(vep):
        vep = re.sub(r"^glm_", "", vep)
        vep = re.sub(r"_score$", "", vep)
        return vep.replace("___", "-").replace("__", "-").replace("_", "-")

    selected_df["Cleaned_VEP"] = selected_df["VEP"].apply(clean_vep_name)

    selected_df["PlotCategory"] = selected_df["PlotCategory"]
    selected_df["Color"] = selected_df["PlotCategory"].map(color_map).fillna("white")

    selected_df = selected_df.sort_values(score_col, ascending=True).reset_index(drop=True)

    bar_height_ref = 0.5
    gap_ref = 1.0 - bar_height_ref
    gap = gap_ref * gap_frac
    bar_height = bar_height_ref

    step = bar_height + gap
    n_bars = len(selected_df)
    y_pos = np.arange(n_bars) * step

    fig_height = max(2.0, fig_height_per_bar * n_bars * (step / 1.0))

    fig, ax = plt.subplots(figsize=(6, fig_height))

    left = 0.5 if score_col == "AUC" else 0.0
    widths = selected_df[score_col] - (0.5 if score_col == "AUC" else 0.0)

    ax.barh(
        y=y_pos,
        width=widths,
        left=left,
        color=selected_df["Color"],
        height=bar_height,
        edgecolor="black",
        linewidth=linewidth,
    )

    if std_col in selected_df.columns:
        x_vals = selected_df[score_col].values
        x_err = selected_df[std_col].values

        ax.errorbar(
            x=x_vals,
            y=y_pos,
            xerr=x_err,
            fmt="none",
            ecolor="black",
            elinewidth=linewidth,
            capsize=0,
            alpha=1,
        )

    ax.set_yticks(y_pos)
    ax.set_yticklabels(selected_df["Cleaned_VEP"])
    ax.set_title(plot_title, fontsize=15, pad=20)
    ax.set_xlabel(x_label, fontsize=13)
    ax.tick_params(axis="y", labelsize=11)
    ax.tick_params(axis="x", labelsize=13)

    outer_pad = bar_height * outer_pad_frac

    ymin = y_pos.min() - (bar_height / 2) - outer_pad
    ymax = y_pos.max() + (bar_height / 2) + outer_pad
    ax.set_ylim(ymin, ymax)

    ax.set_xlim([0.5 if score_col == "AUC" else 0.0, len_x])

    if score_col == "RankScore":
        formatter = mpl.ticker.FuncFormatter(lambda v, pos: f"{v*100:.0f}")
        ax.xaxis.set_major_formatter(formatter)

    always_keep_cleaned = {clean_vep_name(t) for t in always_keep}
    for tick, label in zip(ax.get_yticklabels(), selected_df["Cleaned_VEP"]):
        tick.set_fontweight(
            "bold" if bold_keep_veps and label in always_keep_cleaned else "normal"
        )

    plt.tight_layout()
    output_figure_path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(output_figure_path, dpi=300)
    plt.close(fig)
    print(f"Filtered figure saved to: {output_figure_path}")


In [6]:
generate_filtered_benchmark_figure("functional", len_x = 0.95)
generate_filtered_benchmark_figure("clinical", len_x=1.0)
generate_filtered_benchmark_figure("dd", len_x=0.90)
generate_filtered_benchmark_figure("cancer", len_x=0.95)
generate_filtered_benchmark_figure("mave", len_x=1)
generate_filtered_benchmark_figure("proteingym", len_x=1)

Filtered figure saved to: ..\results\figures\benchmarks\functional_filtered_performance_plot.png
Filtered figure saved to: ..\results\figures\benchmarks\clinical_filtered_performance_plot.png
Filtered figure saved to: ..\results\figures\benchmarks\dd_filtered_performance_plot.png
Filtered figure saved to: ..\results\figures\benchmarks\cancer_filtered_performance_plot.png
Filtered figure saved to: ..\results\figures\benchmarks\mave_filtered_performance_plot.png
Filtered figure saved to: ..\results\figures\benchmarks\proteingym_filtered_performance_plot.png


#### Generate functional vs clinical performance dot plot

In [7]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import re
import matplotlib as mpl
from matplotlib.patches import Patch
from matplotlib import font_manager as fm
from pathlib import Path

font_path = "../resources/fonts/InterVariable.ttf"
fm.fontManager.addfont(font_path)
prop = fm.FontProperties(fname=font_path)
mpl.rcParams['font.family'] = prop.get_name()

def generate_functional_vs_clinical_performance_figure(
    remove_clinvep=True,
    linewidth=1,
    point_size=20,
    alpha=0.9,
    include_legend=True,
    label_fontsize=6,
):
    clinical_file = "../results/benchmarks/clinical_benchmark.txt"
    functional_file = "../results/benchmarks/functional_benchmark.txt"

    x_score_col = "AUC"
    y_score_col = "AUC"

    x_label_name = "Clinical testing dataset"
    y_label_name = "Functional testing dataset"
    metric_label = "AUC"

    features_file = "../resources/feature_lists/all_columns.txt"
    output_base = Path(
        "../results/figures/benchmarks/functional_vs_clinical_performance_scatter"
    )

    df_x = pd.read_csv(clinical_file, sep="\t")
    df_y = pd.read_csv(functional_file, sep="\t")

    df_x = df_x.dropna(subset=[x_score_col]).copy()
    df_y = df_y.dropna(subset=[y_score_col]).copy()

    features_df = pd.read_csv(features_file, sep="\t")
    if "Name" not in features_df.columns:
        raise ValueError("Expected 'Name' column in feature metadata file.")
    if "Category" not in features_df.columns:
        raise ValueError("Expected 'Category' column in feature metadata file.")

    features_df = features_df.rename(columns={"Name": "VEP"})

    df_x_sub = df_x[["VEP", x_score_col]].rename(columns={x_score_col: "x_score"})
    df_y_sub = df_y[["VEP", y_score_col]].rename(columns={y_score_col: "y_score"})

    merged = (
        df_x_sub
        .merge(df_y_sub, on="VEP", how="inner")
        .merge(features_df[["VEP", "Category"]], on="VEP", how="left")
    )

    if remove_clinvep:
        merged = merged[~merged["VEP"].str.startswith("ClinVEP_")].copy()

    merged = merged.dropna(subset=["x_score", "y_score"]).copy()

    if merged.empty:
        raise ValueError(
            "No overlapping VEPs between clinical and functional after filtering."
        )

    category_rename = {
        "FuncVEP": "FuncVEP",
        "Clinical-Trained Meta Predictor": "Clinical-trained",
        "Clinical-Trained Single Predictor": "Clinical-trained",
        "Population-Free": "Population-free",
        "Population-Tuned": "Population-tuned",
    }
    merged["PlotCategory"] = merged["Category"].map(category_rename).fillna(
        merged["Category"]
    )

    color_map = {
        "FuncVEP": "#b21e35",
        "Clinical-trained": "goldenrod",
        "Population-free": "#98e5a5",
        "Population-tuned": "#23a4a6",
    }
    merged["Color"] = merged["PlotCategory"].map(color_map).fillna("white")

    def clean_vep_name(vep):
        vep = re.sub(r"^glm_", "", vep)
        vep = re.sub(r"_score$", "", vep)
        return vep.replace("___", "-").replace("__", "-").replace("_", "-")

    merged["Label"] = merged["VEP"].apply(clean_vep_name)

    x_vals = merged["x_score"].values
    y_vals = merged["y_score"].values
    colors = merged["Color"].values

    fig, ax = plt.subplots(figsize=(5, 5))

    ax.scatter(
        x_vals,
        y_vals,
        s=point_size,
        c=colors,
        alpha=alpha,
        edgecolor="black",
        linewidth=0.3,
        zorder=2,
    )

    line_handles = []
    line_labels = []

    for cat in ["Population-free", "Clinical-trained"]:
        mask = merged["PlotCategory"] == cat
        if mask.sum() >= 2:
            x_cat = merged.loc[mask, "x_score"].values
            y_cat = merged.loc[mask, "y_score"].values

            x_rank = pd.Series(x_cat).rank(method="average")
            y_rank = pd.Series(y_cat).rank(method="average")
            r = np.corrcoef(x_rank, y_rank)[0, 1]

            # Simple linear fit (y = a*x + b)
            a, b = np.polyfit(x_cat, y_cat, 1)
            x_line = np.linspace(x_vals.min(), x_vals.max(), 100)
            y_line = a * x_line + b

            (h_line,) = ax.plot(
                x_line,
                y_line,
                color=color_map.get(cat, "black"),
                linewidth=linewidth * 1.5,
                zorder=1,
            )
            line_handles.append(h_line)
            line_labels.append(f"{cat} (r={r:.2f})")

    def _auc_limits(vals):
        pad = 0.02
        vmin = float(vals.min())
        vmax = float(vals.max())
        lower = max(0.5, vmin - pad)
        upper = min(1.0, vmax + pad)
        if lower == upper:
            lower -= 0.01
            upper += 0.01
        return lower, upper

    x_lower, x_upper = _auc_limits(x_vals)
    y_lower, y_upper = _auc_limits(y_vals)

    ax.set_xlim(x_lower, x_upper)
    ax.set_ylim(y_lower, y_upper)

    x_label = f"{x_label_name} ({metric_label})"
    y_label = f"{y_label_name} ({metric_label})"

    ax.set_xlabel(x_label)
    ax.set_ylabel(y_label)

    title = f"{y_label_name} vs {x_label_name} performance"
    ax.set_title(title)

    if include_legend:
        present_cats = [
            c for c in color_map.keys() if c in merged["PlotCategory"].values
        ]
        cat_handles = [
            Patch(
                facecolor=color_map[c],
                edgecolor="black",
                linewidth=0.5,
                label=c,
            )
            for c in present_cats
        ]

        handles = cat_handles + line_handles
        labels = [c for c in present_cats] + line_labels

        if handles:
            ax.legend(handles, labels, fontsize=8, frameon=True, edgecolor="gray")

    plt.tight_layout()
    output_base.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(f"{output_base}.png", dpi=300)

    x_offset = 0.002
    y_offset = 0.002

    for _, row in merged.iterrows():
        ax.text(
            row["x_score"] + x_offset,
            row["y_score"] + y_offset,
            row["Label"],
            fontsize=label_fontsize,
            ha="left",
            va="bottom",
            zorder=3,  # labels above everything
        )

    fig.savefig(f"{output_base}_labelled.png", dpi=300)
    plt.close(fig)

    print(
        f"Performance dot plots saved to: {output_base}.png and {output_base}_labelled.png"
    )

generate_functional_vs_clinical_performance_figure()

Performance dot plots saved to: ..\results\figures\benchmarks\functional_vs_clinical_performance_scatter.png and ..\results\figures\benchmarks\functional_vs_clinical_performance_scatter_labelled.png


#### Generate mean rank percentile table and figures
This section summarizes overall predictor performance across benchmarks by converting each predictor’s result within each benchmark into a rank percentile (its rank among all predictors included in that benchmark, expressed on a 0–1 scale where lower is better). We then compute a predictor’s mean rank percentile across benchmarks. Because we evaluate MAVE/DMS agreement using two partially overlapping resources (ProteinGym and the MAVE datasets), we first average a predictor’s ProteinGym and MAVE rank percentiles to obtain a single MAVE/DMS percentile, and then average this MAVE/DMS percentile together with the clinical, functional, developmental disorder de novo, and cancer percentiles. This ensures that MAVE/DMS agreement contributes once to the overall summary rather than being double-counted.

In [8]:
import pandas as pd
import re
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import matplotlib as mpl
from matplotlib.patches import Patch
from matplotlib import font_manager as fm

font_path = "../resources/fonts/InterVariable.ttf"
fm.fontManager.addfont(font_path)
prop = fm.FontProperties(fname=font_path)
mpl.rcParams["font.family"] = prop.get_name()

def compute_vep_rank_percentile_summary(remove_clinvep: bool = True,
                                         filter_glm: bool = False):

    dataset_configs = [
        ("clinical", "../results/benchmarks/clinical_benchmark.txt", "AUC"),
        ("functional", "../results/benchmarks/functional_benchmark.txt", "AUC"),
        ("mave", "../results/benchmarks/mave_benchmark.txt", "RankScore"),
        ("proteingym", "../results/benchmarks/proteingym_benchmark.txt", "RankScore"),
        ("cancer", "../results/benchmarks/cancer_benchmark.txt", "AUC"),
        ("dd", "../results/benchmarks/dd_benchmark.txt", "AUC"),
    ]

    features_file = "../resources/feature_lists/all_columns.txt"

    features_df = pd.read_csv(features_file, sep="\t")
    if "Name" not in features_df.columns:
        raise ValueError("Expected 'Name' column in feature metadata file.")
    if "Category" not in features_df.columns:
        raise ValueError("Expected 'Category' column in feature metadata file.")

    features_df = features_df.rename(columns={"Name": "VEP"})

    category_rename = {
        "FuncVEP": "FuncVEP",
        "Clinical-Trained Meta Predictor": "Clinical-trained",
        "Clinical-Trained Single Predictor": "Clinical-trained",
        "Population-Free": "Population-free",
        "Population-Tuned": "Population-tuned",
    }

    def compute_dataset_percentiles(dataset_name, file_path, score_col):
        df = pd.read_csv(file_path, sep="\t")

        if "VEP" not in df.columns:
            raise ValueError(f"'VEP' column not found in {file_path}")
        if score_col not in df.columns:
            raise ValueError(f"'{score_col}' column not found in {file_path}")

        df = df.dropna(subset=[score_col]).copy()

        # Optional: exclude ClinVEP
        if remove_clinvep:
            df = df[~df["VEP"].str.startswith("ClinVEP_")].copy()

        if filter_glm:
            glm_mask = df["VEP"].str.startswith("glm_")
            if glm_mask.any():
                best_glm = df.loc[glm_mask].nlargest(1, score_col)
                glm_to_keep = set(best_glm["VEP"].unique())
                df = df[(~glm_mask) | (df["VEP"].isin(glm_to_keep))].copy()

        if df.empty:
            return pd.DataFrame(columns=["VEP", f"{dataset_name}_pct"])

        df["rank"] = df[score_col].rank(ascending=False, method="average")
        n = len(df)
        if n > 1:
            df[f"{dataset_name}_pct"] = (df["rank"] - 1) / (n - 1)
        else:
            df[f"{dataset_name}_pct"] = 0.0

        return df[["VEP", f"{dataset_name}_pct"]]

    per_vep = None
    dataset_to_col = {}

    for dataset_name, file_path, score_col in dataset_configs:
        col_name = f"{dataset_name}_pct"
        dataset_to_col[dataset_name] = col_name

        ds_df = compute_dataset_percentiles(dataset_name, file_path, score_col)

        if per_vep is None:
            per_vep = ds_df
        else:
            per_vep = per_vep.merge(ds_df, on="VEP", how="outer")

    if per_vep is None or per_vep.empty:
        raise RuntimeError("No percentile data available. Check benchmark files.")

    dms_cols = ["mave_pct", "proteingym_pct"]
    for c in dms_cols:
        if c not in per_vep.columns:
            per_vep[c] = pd.NA

    per_vep["DMS_percentile"] = per_vep[dms_cols].mean(axis=1, skipna=True)

    group_cols = [
        "clinical_pct",
        "functional_pct",
        "DMS_percentile",
        "cancer_pct",
        "dd_pct",
    ]
    for c in group_cols:
        if c not in per_vep.columns:
            per_vep[c] = pd.NA

    per_vep["Mean_percentile"] = per_vep[group_cols].mean(axis=1, skipna=True)

    def participated_datasets(row):
        present = []
        for ds_name, col in dataset_to_col.items():
            if col in row and pd.notna(row[col]):
                present.append(ds_name)
        return ";".join(present)

    per_vep["Datasets"] = per_vep.apply(participated_datasets, axis=1)

    def clean_vep_name(vep):
        vep = re.sub(r"^glm_", "", vep)
        vep = re.sub(r"_score$", "", vep)
        return vep.replace("___", "-").replace("__", "-").replace("_", "-")

    per_vep = per_vep.merge(features_df[["VEP", "Category"]], on="VEP", how="left")
    per_vep["PlotCategory"] = per_vep["Category"].map(category_rename).fillna(per_vep["Category"])
    per_vep["VEP_clean"] = per_vep["VEP"].apply(clean_vep_name)

    return per_vep


In [9]:
def save_vep_rank_percentile_summary(remove_clinvep: bool = True,
                                      filter_glm: bool = False):
    per_vep = compute_vep_rank_percentile_summary(
        remove_clinvep=remove_clinvep,
        filter_glm=filter_glm,
    )

    rename_cols = {
        "clinical_pct": "Clinical percentile",
        "functional_pct": "Functional percentile",
        "DMS_percentile": "DMS percentile",
        "cancer_pct": "Cancer percentile",
        "dd_pct": "DD percentile",
        "Mean_percentile": "Mean percentile",
    }
    per_vep = per_vep.rename(columns=rename_cols)

    ordered_cols = [
        "VEP_clean",
        "VEP",
        "PlotCategory",
        "Mean percentile",
        "Clinical percentile",
        "Functional percentile",
        "DMS percentile",
        "Cancer percentile",
        "DD percentile",
        "Datasets",
    ]
    ordered_cols = [c for c in ordered_cols if c in per_vep.columns]
    per_vep = per_vep[ordered_cols]

    if "Mean percentile" in per_vep.columns:
        per_vep = per_vep.sort_values(
            by="Mean percentile", ascending=True, na_position="last"
        )

    output_file = Path("../results/tables/benchmarks/vep_average_rank_percentile.txt")
    output_file.parent.mkdir(parents=True, exist_ok=True)
    per_vep.to_csv(output_file, sep="\t", index=False)
    print(f"VEP average rank percentile table saved to: {output_file}")

    return per_vep


In [10]:
def generate_rank_percentile_figures(
    remove_clinvep: bool = True,
    filter_glm: bool = False,
    linewidth: float = 0.5,
    gap_frac: float = 0.25,
    outer_pad_frac: float = 0.9,
    veps_per_category: int = 3,
):
    df = compute_vep_rank_percentile_summary(
        remove_clinvep=remove_clinvep,
        filter_glm=filter_glm,
    )

    df = df.dropna(subset=["Mean_percentile"]).copy()
    if df.empty:
        raise ValueError("No VEPs with valid mean percentile for plotting.")

    color_map = {
        "FuncVEP": "#b21e35",
        "Clinical-trained": "goldenrod",
        "Population-free": "#98e5a5",
        "Population-tuned": "#23a4a6",
    }
    df["Color"] = df["PlotCategory"].map(color_map).fillna("white")

    TARGET_ASPECT = 5.73 / 4.3

    def _plot(
        df_plot,
        output_path: Path,
        title: str,
        x_max: float,
        include_legend: bool,
        bold_keep_veps: bool,
        bar_height_local: float,
        fig_width: float,
        fig_height_per_bar: float,
        title_fontsize: float,
        xlabel_fontsize: float,
        x_tick_fontsize: float,
        y_tick_fontsize: float,
        linewidth_local: float,
        outer_pad_frac_local: float,
        gap_ref_local: float,
        match_filtered_aspect: bool = False,
    ):
        df_plot = df_plot.sort_values("Mean_percentile", ascending=True).reset_index(drop=True)

        n_bars = len(df_plot)
        gap_ref = gap_ref_local
        gap = gap_ref * gap_frac
        step = bar_height_local + gap
        y_pos = np.arange(n_bars) * step
        outer_pad = max(0.2, bar_height_local * outer_pad_frac_local)

        fig_height = max(2.0, fig_height_per_bar * n_bars * (step / 1.0))

        if match_filtered_aspect:
            fig_width = TARGET_ASPECT * fig_height

        fig, ax = plt.subplots(figsize=(fig_width, fig_height))

        widths = df_plot["Mean_percentile"].values
        lefts = np.zeros_like(widths)

        ax.barh(
            y=y_pos,
            width=widths,
            left=lefts,
            color=df_plot["Color"],
            height=bar_height_local,
            edgecolor="black",
            linewidth=linewidth_local,
        )

        ax.set_xlim([0.0, x_max])
        formatter = mpl.ticker.FuncFormatter(lambda v, pos: f"{v*100:.0f}")
        ax.xaxis.set_major_formatter(formatter)

        ax.set_yticks(y_pos)
        ax.set_yticklabels(df_plot["VEP_clean"])
        ax.set_ylim(y_pos.min() - outer_pad, y_pos.max() + outer_pad)

        ax.invert_yaxis()

        ax.set_xlabel("Mean rank percentile (%)", fontsize=xlabel_fontsize)
        ax.set_title(title, fontsize=title_fontsize, loc="center", pad=20)
        ax.tick_params(axis="x", labelsize=x_tick_fontsize)
        ax.tick_params(axis="y", labelsize=y_tick_fontsize)

        if include_legend:
            present_cats = [c for c in color_map.keys() if c in df_plot["PlotCategory"].values]
            handles = [
                Patch(facecolor=color_map[c], label=c, edgecolor="black", linewidth=linewidth_local)
                for c in present_cats
            ]
            if handles:
                ax.legend(
                    handles=handles,
                    loc="upper right",
                    fontsize=y_tick_fontsize + 2,
                    frameon=True,
                    edgecolor="gray",
                )

        if bold_keep_veps:
            always_keep = {
                "FuncVEP_CTI", "FuncVEP_CTE", "FuncVEP_SP",
                "ClinVEP_CTI", "ClinVEP_CTE", "ClinVEP_SP",
            }

            def clean_vep_name(vep):
                vep = re.sub(r"^glm_", "", vep)
                vep = re.sub(r"_score$", "", vep)
                return vep.replace("___", "-").replace("__", "-").replace("_", "-")

            always_keep_cleaned = {clean_vep_name(t) for t in always_keep}
            for tick, label in zip(ax.get_yticklabels(), df_plot["VEP_clean"]):
                tick.set_fontweight(
                    "bold" if label in always_keep_cleaned else "normal"
                )

        plt.tight_layout()
        output_path.parent.mkdir(parents=True, exist_ok=True)
        fig.savefig(output_path, dpi=300)
        plt.close(fig)
        print(f"Rank percentile figure saved to: {output_path}")

    full_output = Path(
        "../results/figures/benchmarks/vep_mean_rank_percentile_plot.png"
    )
    _plot(
        df_plot=df,
        output_path=full_output,
        title="Average rank percentile across benchmarks",
        x_max=1.0,
        include_legend=True,
        bold_keep_veps=False,
        bar_height_local=0.3,
        fig_width=6.0,
        fig_height_per_bar=0.28,
        title_fontsize=8,
        xlabel_fontsize=7,
        x_tick_fontsize=6,
        y_tick_fontsize=4.5,
        linewidth_local=linewidth,
        outer_pad_frac_local=outer_pad_frac,
        gap_ref_local=0.5,
        match_filtered_aspect=False,
    )

    always_keep = {
        "FuncVEP_CTI", "FuncVEP_CTE", "FuncVEP_SP",
        "ClinVEP_CTI", "ClinVEP_CTE", "ClinVEP_SP",
    }

    remaining_df = df[~df["VEP"].isin(always_keep)].copy()

    best_per_category = (
        remaining_df.sort_values("Mean_percentile", ascending=True)
        .groupby("PlotCategory", dropna=False)
        .head(veps_per_category)
    )

    filtered_df = pd.concat(
        [
            df[df["VEP"].isin(always_keep)],
            best_per_category,
        ]
    ).drop_duplicates(subset="VEP")

    filtered_output = Path(
        "../results/figures/benchmarks/vep_mean_rank_percentile_filtered_plot.png"
    )
    _plot(
        df_plot=filtered_df,
        output_path=filtered_output,
        title="Average rank percentile across benchmarks",
        x_max=0.35,
        include_legend=False,
        bold_keep_veps=True,
        bar_height_local=0.5,
        fig_width=6.0,
        fig_height_per_bar=0.7,
        title_fontsize=17,
        xlabel_fontsize=15,
        x_tick_fontsize=15,
        y_tick_fontsize=13,
        linewidth_local=1.0,
        outer_pad_frac_local=0.9,
        gap_ref_local=1.0 - 0.5,
        match_filtered_aspect=True,
    )


In [11]:
save_vep_rank_percentile_summary()
generate_rank_percentile_figures()

VEP average rank percentile table saved to: ..\results\tables\benchmarks\vep_average_rank_percentile.txt
Rank percentile figure saved to: ..\results\figures\benchmarks\vep_mean_rank_percentile_plot.png
Rank percentile figure saved to: ..\results\figures\benchmarks\vep_mean_rank_percentile_filtered_plot.png
